# 07. Land Value Capture (LVC) Szimuláció

**Cél**: A közösségi infrastruktúra-fejlesztések (Mázsa tér intermodális csomópont, vasúti átjárók, zöldfelületi rehabilitáció) által generált magánvagyon-növekmény visszanyerési mechanizmusainak szimulációja.

**Módszertan**: Dinamikus Cash Flow (DCF), Net Present Value (NPV), Belső Megtérülési Ráta (IRR) és érzékenységvizsgálat a fejlesztési szintek (Tier 1, 2, 3) és az értéknövekmény-elvonási kulcsok (Capture Rate) függvényében.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np

df = load_szamitott_master()
elado = df[df['listing_type'] == 'elado'].copy()

# Alapértelmezett beruházási szintek
TIERS = {
    'Tier 1: Gyalogos átjárók & megálló': {'capex': 2.5e9, 'evek': 2, 'premium_pct': 0.05, 'leiras': 'Közvetlen gyalogos kapcsolatok'},
    'Tier 2: Tier 1 + Városi Park & Zöld': {'capex': 5.0e9, 'evek': 3, 'premium_pct': 0.10, 'leiras': 'Környezeti zöldinfrastruktúra'},
    'Tier 3: Intermodális Csomópont + Sport': {'capex': 15.0e9, 'evek': 5, 'premium_pct': 0.20, 'leiras': 'Komplex városmegújítás'}
}
print("LVC modell inicializálva.")

LVC modell inicializálva.


### 1. LVC Beruházási Szcenáriók és Pénzügyi KPI-k
A Tier 2 fejlesztési szint (5 Mrd Ft CAPEX) hatása a közvetlen hatásterület ingatlanvagyonára (20%-os capture rate mellett).

In [2]:
# Érintett ingatlanállomány becslése a Mázsa tér 15 perces izokrónjában
erintett_lakasok_becsles = 12000  # becsült lakásszám a körzetben
atlag_lakasar = elado['price_huf'].median()
erintett_vagyon = erintett_lakasok_becsles * atlag_lakasar

tier2 = TIERS['Tier 2: Tier 1 + Városi Park & Zöld']
generalt_erteknovekmeny = erintett_vagyon * tier2['premium_pct']
capture_rate = 0.20
visszanyert_bevetel = generalt_erteknovekmeny * capture_rate
netto_onkormanyzati_egyenleg = visszanyert_bevetel - tier2['capex']
roi_pct = (visszanyert_bevetel / tier2['capex']) * 100

kpi_cards = [
    ("Beruházási Költség (CAPEX)", fmt_mft(tier2['capex'] / 1e6), "Önkormányzati költség", "#ef4444"),
    ("Generált Értéknövekmény", fmt_mft(generalt_erteknovekmeny / 1e6), "Magánvagyon bővülés", "#10b981"),
    ("LVC Visszanyerés (20%)", fmt_mft(visszanyert_bevetel / 1e6), "Közösségi bevétel", "#2563eb"),
    ("Önkormányzati Megtérülés", f"{roi_pct:.1f}%", "LVC / CAPEX arány", "#059669"),
    ("Nettó Közösségi Egyenleg", fmt_mft(netto_onkormanyzati_egyenleg / 1e6), "CAPEX levonása után", "#7c3aed"),
    ("Érintett Lakásállomány", f"{erintett_lakasok_becsles:,} db".replace(',', ' '), "15p sétaövezet", "#d97706")
]
display(HTML(kpi_grid_html(kpi_cards)))

### 2. Kumulált Készpénzáramlási Pálya (20 Éves Horizont)
A projekt kumulált cash flow-ja a beruházási időszak alatt és a fejlesztési hozzájárulások befolyása után.

In [3]:
evek = np.arange(0, 21)
cf_alap = np.zeros(21)
cf_alap[1:4] = -tier2['capex'] / 3.0  # 3 éves beruházás
# Éves LVC bevételek az 4. évtől fokozatosan 15 éven át
cf_alap[4:19] = visszanyert_bevetel / 15.0

cum_cf = np.cumsum(cf_alap)

fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=evek, y=cum_cf / 1e6,
    mode='lines+markers',
    name='Kumulált Cash Flow (Alap szcenárió)',
    line=dict(color='#2563eb', width=3),
    marker=dict(size=8)
))
fig1.add_hline(y=0, line_dash='dash', line_color='red', annotation_text='Megtérülési Küszöb (Break-even)')
fig1.update_layout(
    title='LVC Beruházás Kumulált Pénzárama (Millió Ft)',
    xaxis_title='Évek',
    yaxis_title='Kumulált Egyenleg (M Ft)',
    template=PLOTLY_TEMPLATE,
    height=450
)
fig1.show()

### 3. Érzékenységvizsgálat (Diszkontráta vs. Capture Rate Mátrix)
Az NPV alakulása a diszkontráta (3% - 8%) és a visszanyerési kulcs (10% - 35%) kombinációiban.

In [4]:
rates = [0.03, 0.04, 0.05, 0.06, 0.07, 0.08]
captures = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35]

npv_matrix = np.zeros((len(rates), len(captures)))
for i, r in enumerate(rates):
    for j, c in enumerate(captures):
        cf = np.zeros(21)
        cf[1:4] = -tier2['capex'] / 3.0
        cf[4:19] = (generalt_erteknovekmeny * c) / 15.0
        npv = sum(cf[t] / ((1 + r) ** t) for t in range(len(cf)))
        npv_matrix[i, j] = npv / 1e6

fig2 = px.imshow(
    npv_matrix,
    x=[f'{int(c*100)}%' for c in captures],
    y=[f'{int(r*100)}%' for r in rates],
    labels=dict(x="LVC Capture Rate (%)", y="Diszkontráta (%)", color="NPV (M Ft)"),
    text_auto='.0f',
    color_continuous_scale='RdYlGn',
    title='LVC Projekt Nettó Jelenértéke (NPV, M Ft) Érzékenységi Mátrixban',
    template=PLOTLY_TEMPLATE
)
fig2.update_layout(height=420)
fig2.show()

### 4. Interaktív LVC Szimulációs Modell
Dinamikusan tesztelhetők a különböző beruházási szintek és pénzügyi paraméterek.

In [5]:
w_tier = widgets.Dropdown(options=list(TIERS.keys()), value='Tier 2: Tier 1 + Városi Park & Zöld', description='Szint:')
w_cap = widgets.FloatSlider(min=0.05, max=0.40, step=0.05, value=0.20, description='Capture %:')
w_disc = widgets.FloatSlider(min=0.02, max=0.10, step=0.01, value=0.05, description='Diszkont %:')

out_lvc = widgets.Output()

def frissit_lvc(*args):
    t_info = TIERS[w_tier.value]
    gain = erintett_vagyon * t_info['premium_pct']
    rec = gain * w_cap.value
    cf = np.zeros(21)
    evek = t_info['evek']
    cf[1:evek+1] = -t_info['capex'] / evek
    cf[evek+1:evek+16] = rec / 15.0
    npv_val = sum(cf[t] / ((1 + w_disc.value) ** t) for t in range(len(cf)))
    
    with out_lvc:
        clear_output(wait=True)
        res_kpis = [
            ("Szimulált NPV", fmt_mft(npv_val / 1e6), "Diszkontált egyenleg", "#10b981" if npv_val > 0 else "#ef4444"),
            ("Visszanyerés Összesen", fmt_mft(rec / 1e6), "15 év alatt", "#2563eb"),
            ("CAPEX Fedezet", f"{(rec / t_info['capex']) * 100:.1f}%", "Megtérülési arány", "#7c3aed")
        ]
        display(HTML(kpi_grid_html(res_kpis)))

w_tier.observe(frissit_lvc, names='value')
w_cap.observe(frissit_lvc, names='value')
w_disc.observe(frissit_lvc, names='value')

display(widgets.HBox([w_tier, w_cap, w_disc]))
display(out_lvc)
frissit_lvc()

Output()